# Adaptive Minimax (Connect Four);

In [ ]:
# 1) upload the zip file given with the poject folder
import os, sys, glob, zipfile
from google.colab import files
up = files.upload()                      # choose the zip in the dialog
zname = next(iter(up))
with zipfile.ZipFile(zname) as z:
    z.extractall("/content/project")
hits = glob.glob("/content/project/**/serve.py", recursive=True)
assert hits, "serve.py not found - did you upload the right zip?"
ROOT = os.path.dirname(hits[0])
os.chdir(ROOT); sys.path.insert(0, ROOT)
print("Project folder:", ROOT)
print(sorted(os.listdir(ROOT)))

In [ ]:
# 2) Install requirements and run the self-check
!pip -q install -r requirements.txt
!python check_setup.py

In [ ]:
# 3) Correctness tests (27 tests, ~1 minute)
!python tests/run_tests.py 2>&1 | tail -8

In [ ]:
# 4) Start the web demo 
import subprocess, time, socket
from google.colab import output

PORT = 8000
proc = subprocess.Popen([sys.executable, "serve.py", "--no-browser", "--port", str(PORT)],
                        stdout=open("serve.log", "w"), stderr=subprocess.STDOUT, cwd=ROOT)
for _ in range(180):                      # wait until the server answers
    try:
        socket.create_connection(("127.0.0.1", PORT), timeout=1).close(); break
    except OSError:
        time.sleep(1)
else:
    print(open("serve.log").read()); raise RuntimeError("server did not start - see log above")
print("Server is up.")
output.serve_kernel_port_as_iframe(PORT, height=900)   
# Prefer a full browser tab?  Uncomment:
# output.serve_kernel_port_as_window(PORT)

In [ ]:
# 5) (optional) No browser needed: ask all six agents for their move in any position
from adaptive_minimax.experiments import load_artifacts, make_agents
from adaptive_minimax.game import State, HEIGHT, WIDTH, H1

agents = make_agents(load_artifacts())

def show(moves):
    s = State()
    for m in moves: s = s.play(m)
    first = (s.n % 2 == 0)
    p1, p2 = (s.cur, s.opp) if first else (s.opp, s.cur)
    for r in range(HEIGHT - 1, -1, -1):
        print(" ".join("X" if p1 >> (c*H1+r) & 1 else "O" if p2 >> (c*H1+r) & 1 else "." for c in range(WIDTH)))
    print("0 1 2 3 4 5 6"); return s

history = [3, 3, 4]          # <- edit: columns played so far (0-6)
budget  = 1000               # <- nodes per move
s = show(history)
for k, ag in agents.items():
    move, info = ag.choose(s, budget)
    print(f"{ag.label:28s} -> column {move}   nodes={info['nodes']}")

### Notes
* Full re-run of the research pipeline: `!python scripts/01_baselines.py` ... `!python scripts/05_make_plots.py` (see README). The quick versions take a few minutes.